In [31]:
import pandas as pd

In [32]:
DATA_PATH = "D:\Sesi 11\Session 11\jobpostingdata.csv"
MODEL_PATH = "model.pkl"

<>:1: SyntaxWarning: invalid escape sequence '\S'
<>:1: SyntaxWarning: invalid escape sequence '\S'
C:\Users\prk\AppData\Local\Temp\ipykernel_16456\4063265181.py:1: SyntaxWarning: invalid escape sequence '\S'
  DATA_PATH = "D:\Sesi 11\Session 11\jobpostingdata.csv"


In [33]:
df = pd.read_csv(DATA_PATH)
df.head()

,title,fraudulent,text
0,PHP Developer,0,PHP Developer You're a skilled developer. You ...
1,CUSTOMER SERVICE AGENT,1,CUSTOMER SERVICE AGENT Aegis is a global busi...
2,VP Marketing & Growth,0,VP Marketing & Growth Depop is an exciting new...
3,SAP BW Developer/Architect,1,SAP BW Developer/Architect Assist with Full L...
4,Administrative Assistant,1,Administrative Assistant With decades of exper...


In [34]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.classify.naivebayes import NaiveBayesClassifier
from nltk.classify.util import accuracy

In [35]:
def get_tag(tag:str):
    
    if tag[0].lower() in ['v', 'r', 'n']:
        return tag[0].lower()
    elif tag[0].lower() == 'j':
        return 'a'
    else:
        return 'n'
    
def lemmatization(tokens):
    
    lemmatizer = WordNetLemmatizer()
    
    tagged = pos_tag(tokens)
    lemmatized = []
    
    for token, tag in tagged:
        base = lemmatizer.lemmatize(token, get_tag(tag))
        lemmatized.append(base)
        
    return lemmatized

def text_preprocessor(sentence : str):
    
    eng_sw = stopwords.words('english')
    
    tokens = word_tokenize(sentence)
    clean = [tokens.lower() for token in tokens if token in token.isalpha() and token not in eng_sw]
    lemmatized = lemmatization(clean)
    
    return lemmatized
    

In [36]:
# existed_words -> semua kata unik yang ada di corpus
# return value -> {word1: true, word2: false}

def feature_extraction(sentence, existed_words):
    tokens = set(text_preprocessor(sentence))
    
    feature = {}
    
    for word in existed_words:
        feature[word] = (word in tokens)
        
    return feature   

In [37]:
from sklearn.model_selection import train_test_split

In [38]:
def train_classifier():
    X = df['text']
    Y = df['fraudulent']
    
    corpus = " ".join(X)
    
    existed_words = set(text_preprocessor(corpus))
    
    features = [(feature_extraction(sentence), cat) for sentence, cat in zip(X, Y)]
    
    train_data, test_data = train_test_split(
        features, test_size=0.2
    )
    
    classifier = NaiveBayesClassifier.train(train_data)
    
    print(f"accuracy : {accuracy(classifier, test_data) * 100}%")
    
    return classifier, existed_words

In [39]:
train_classifier()

TypeError: argument of type 'bool' is not iterable

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
def train_vectorizer():
    X = df["text"]
    
    vectorizer = TfidfVectorizer(
        ngram_range=(1,2),
        min_df = 1,
        max_df = 0.9,
        stop_words='english'
    )
    
    job_vectors = vectorizer.fit_transform(X)
    
    return vectorizer, job_vectors

In [ ]:
import os
import pickle

In [ ]:
def load_model():
    if os.path.exists(MODEL_PATH):
        
        with open(MODEL_PATH, "rb") as file:
            classifier, existed_words, vectorizer, vectors = pickle.load(file)
            
            return classifier, existed_words, vectorizer, vectors
        
    else:
        
        print("Model not found")
        print("Training Model!")
        
        classifier, existed_words = train_classifier()
        vectorizer, vectors = train_vectorizer()
        
        with open(MODEL_PATH, "wb") as file:
            pickle.dump((classifier, existed_words, vectorizer, vectors), file)
            
        return classifier, existed_words, vectorizer, vectors

In [ ]:
def print_menu(sentence, cat):
    print("Real / Fake Job System")
    print(f"text: {sentence}")
    print(f"category: {cat}")
    print("="*25)
    print("[1] Write Text")
    print("[2] Recommend")
    print("[3] NER")
    print("[4] Exit")
    
    return input("Please Choose Between 1-4: ")

In [ ]:
def write_sentence():
    sentence = ""
    
    while len(sentence.strip()) < 20 or len(sentence.strip().split(" ")):
        sentence = input("Input Text Min. 20 Char and Min. 3 words")
        
        return sentence

In [ ]:
def classify_text(classifier, sentence: str, existed_words):
    
    feature = feature_extraction(sentence, existed_words)
    
    cat = classifier.classify(feature)
    
    if cat == 1:
        return 'FAKE'
    
    else:
        return "RILL"

In [ ]:
def load_NER(nlp, sentence):
    
    doc = nlp(sentence)
    
    ents_dict = {}
    
    for ent in doc.ents:
        if ent.label_ not in ents_dict.keys():
            ents_dict[ent.label_] = []
            
        ents_dict[ent.label_].append(ent.text)
        
    return ents_dict

In [ ]:
def print_NER(ents_dict):
    if ents_dict is None:
        print("No Text Found!")
        return 
    
    if ents_dict:
        print("NER")
        for key, value in ents_dict.items():
            print(f"{key} : {value}")
    
    else:
        print("No Entities Found")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def recommend_top_n(vectorizer : TfidfVectorizer, job_vectors, query, topn = 5):
    
    vectorized_query = vectorizer.transform([query])
    
    similarity = cosine_similarity(vectorized_query, job_vectors).flatten()
    
    
    
    top_idx = similarity.argsort()[::-1][:5]
    
    for i, idx in enumerate(top_idx, 1):
        print(f"{i}: {df["title"].iloc[idx]}")

In [ ]:
import spacy

In [ ]:
def menu():
    sentence = ""
    cat = ""
    
    # classify
    classifier = None
    existed_words = None
    
    # recommend
    vectorizer = None 
    job_vectors = None
    
    # NER
    nlp = spacy.load("en_core_web_sm")
    ents_dict = None
    
    while True:
        choice = print_menu(sentence, cat).strip()
        
        print()
        
        if choice == 1:
            sentence = write_sentence()
            
            if classifier is None or existed_words is None or vectorizer is None or job_vectors is None:
                classifier, existed_words, vectorizer, job_vectors = load_model()
                
            cat = classify_text(classifier, sentence, existed_words)
             
            ents_dict = load_NER(nlp, sentence)
            
        elif choice == 2:
            
        elif choice == 3:
            
            print_NER(ents_dict)
            
        elif choice == 4:
            print("Thank you!")
            return
        else:
            print("Invalid input!")
        print()